# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Preprocesador para árboles: solo imputación (los árboles no requieren escalado)
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), feature_columns),
    ]
)

# Preprocesador para modelos sensibles a la escala: imputación + estandarización
scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), feature_columns),
    ]
)

print(tree_preprocessor)
print(scaled_preprocessor)


ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 ['alcohol', 'malic_acid', 'ash',
                                  'alcalinity_of_ash', 'magnesium',
                                  'total_phenols', 'flavanoids',
                                  'nonflavanoid_phenols', 'proanthocyanins',
                                  'color_intensity', 'hue',
                                  'od280/od315_of_diluted_wines', 'proline'])])
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['alcohol', 'malic_acid', 'ash',
                                  'alcalinity_of_ash', 'magnesium',
                                  'total_phenols', 'flavanoids',
                                  'nonflavan

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Porque `StandardScaler` **aprende parámetros de los datos** (media y desviación de cada variable). Si se ajusta sobre todo `train` antes de la validación cruzada, en cada fold el fold de validación ya influyó en esa media y desviación: hay **fuga de información (data leakage)** y la métrica de CV resulta optimista. El fold de validación debe simular datos nunca vistos.
>
> Dentro del `Pipeline`, `cross_validate`/`GridSearchCV` llaman `fit` solo sobre los folds de entrenamiento y aplican `transform` al fold de validación con esos parámetros, igual que ocurrirá en producción con el conjunto de test y con datos nuevos. Además, el objeto guardado (`champion_model.joblib`) contiene el escalador ajustado, así que en inferencia basta llamar `predict` sobre datos crudos sin replicar el preprocesamiento a mano.


## 3. Modelos candidatos

In [5]:
models = {
    "DecisionTree": Pipeline([
        ("preprocessor", clone(tree_preprocessor)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "LogisticRegression": Pipeline([
        ("preprocessor", clone(scaled_preprocessor)),
        ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]),
    "KNN": Pipeline([
        ("preprocessor", clone(scaled_preprocessor)),
        ("model", KNeighborsClassifier()),
    ]),
}

for name in models:
    print(name)


DecisionTree
LogisticRegression
KNN


## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipeline in models.items():
    scores = cross_validate(
        pipeline, X_train, y_train,
        cv=cv, scoring="accuracy", return_train_score=True,
    )
    rows.append({
        "model": name,
        "train_accuracy_mean": scores["train_score"].mean(),
        "val_accuracy_mean": scores["test_score"].mean(),
        "val_accuracy_std": scores["test_score"].std(),
        "gap_train_val": scores["train_score"].mean() - scores["test_score"].mean(),
    })

initial_comparison = (pd.DataFrame(rows)
                      .sort_values("val_accuracy_mean", ascending=False)
                      .reset_index(drop=True))
initial_comparison.round(4)


,model,train_accuracy_mean,val_accuracy_mean,val_accuracy_std,gap_train_val
0,LogisticRegression,1.0000,0.9791,0.0277,0.0209
1,KNN,0.9771,0.9507,0.0353,0.0264
2,DecisionTree,1.0000,0.9022,0.0586,0.0978


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "DecisionTree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 3, 5],
        "model__criterion": ["gini", "entropy"],
    },
    "LogisticRegression": {
        "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11, 15],
        "model__weights": ["uniform", "distance"],
    },
}

searches = {}
rows = []
for name, pipeline in models.items():
    search = GridSearchCV(
        pipeline, param_grids[name],
        cv=cv, scoring="accuracy",
        return_train_score=True, n_jobs=-1,
    )
    search.fit(X_train, y_train)
    searches[name] = search

    best = search.cv_results_
    i = search.best_index_
    rows.append({
        "model": name,
        "best_params": search.best_params_,
        "cv_accuracy_mean": search.best_score_,
        "cv_accuracy_std": best["std_test_score"][i],
        "train_accuracy_mean": best["mean_train_score"][i],
    })

tuned_comparison = (pd.DataFrame(rows)
                    .sort_values(["cv_accuracy_mean", "cv_accuracy_std"], ascending=[False, True])
                    .reset_index(drop=True))
pd.set_option("display.max_colwidth", None)
tuned_comparison.round(4)


,model,best_params,cv_accuracy_mean,cv_accuracy_std,train_accuracy_mean
0,LogisticRegression,{'model__C': 1.0},0.9791,0.0277,1.0000
1,KNN,"{'model__n_neighbors': 7, 'model__weights': 'uniform'}",0.9579,0.0405,0.9648
2,DecisionTree,"{'model__criterion': 'gini', 'model__max_depth': 4, 'model__min_samples_leaf': 1}",0.9022,0.0586,0.9982


## 6. Selección y persistencia del campeón

In [8]:
# Campeón: mayor best_score_ de CV; en empate, menor desviación estándar
champion_name = tuned_comparison.loc[0, "model"]
champion = searches[champion_name].best_estimator_   # refit=True: ya re-entrenado con todo train
print("Campeón:", champion_name)
print("Mejores hiperparámetros:", searches[champion_name].best_params_)

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "best_params": {k: (v if not isinstance(v, np.generic) else v.item())
                    for k, v in searches[champion_name].best_params_.items()},
    "cv_accuracy_mean": float(tuned_comparison.loc[0, "cv_accuracy_mean"]),
    "cv_accuracy_std": float(tuned_comparison.loc[0, "cv_accuracy_std"]),
    "cv_folds": cv.get_n_splits(),
    "scoring": "accuracy",
    "random_state": RANDOM_STATE,
    "features": feature_columns,
    "target": contract["target"],
    "n_train": int(len(X_train)),
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)

# Todos los resultados de CV de las tres búsquedas
cv_frames = []
for name, search in searches.items():
    frame = pd.DataFrame(search.cv_results_)
    frame.insert(0, "model", name)
    cv_frames.append(frame)
cv_results = pd.concat(cv_frames, ignore_index=True)
cv_results["params"] = cv_results["params"].astype(str)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

print(json.dumps(training_metadata, indent=2, ensure_ascii=False)[:600])


Campeón: LogisticRegression
Mejores hiperparámetros: {'model__C': 1.0}
{
  "champion": "LogisticRegression",
  "best_params": {
    "model__C": 1.0
  },
  "cv_accuracy_mean": 0.9790640394088669,
  "cv_accuracy_std": 0.027713464392187602,
  "cv_folds": 5,
  "scoring": "accuracy",
  "random_state": 42,
  "features": [
    "alcohol",
    "malic_acid",
    "ash",
    "alcalinity_of_ash",
    "magnesium",
    "total_phenols",
    "flavanoids",
    "nonflavanoid_phenols",
    "proanthocyanins",
    "color_intensity",
    "hue",
    "od280/od315_of_diluted_wines",
    "proline"
  ],
  "target": "target",
  "n_train": 142
}


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> **Campeón: Regresión Logística (`C = 1.0`, con imputación + `StandardScaler` en el pipeline).**
>
> - **Desempeño:** tiene la mayor accuracy de validación cruzada, **0.979** (5 folds estratificados), frente a 0.958 de KNN (k = 7) y 0.902 del árbol (profundidad 4). La búsqueda de hiperparámetros no mejoró su configuración por defecto, lo que indica que el modelo ya era adecuado para el problema.
> - **Variabilidad:** también es la más estable entre folds (desviación **0.028**, frente a 0.041 de KNN y 0.059 del árbol). Con solo 142 muestras de entrenamiento esto importa: el resultado depende menos de qué ejemplos caen en cada fold.
> - **Sobreajuste:** su accuracy de entrenamiento es 1.00, pero la brecha con validación es pequeña (≈ 0.02). En este dataset las clases son casi linealmente separables tras estandarizar, así que acertar todo el entrenamiento no implica memorizar. El árbol sí sobreajusta: brecha ≈ 0.10 sin podar y ≈ 0.10 aún con `max_depth = 4`.
> - **Complejidad:** es un modelo lineal con 3 × 13 coeficientes + 3 interceptos, rápido de entrenar y de predecir. KNN debe guardar todo el conjunto de entrenamiento y calcular distancias en cada predicción.
> - **Interpretabilidad:** los coeficientes, sobre variables estandarizadas, indican cuánto empuja cada variable hacia cada clase, y además entrega **probabilidades calibradas razonablemente**, útiles en la inferencia. El árbol sería más fácil de explicar con reglas, pero pierde casi 8 puntos de accuracy y es el más inestable. Por la multicolinealidad entre las variables fenólicas (ver Challenge 1), los coeficientes individuales deben leerse con cautela; la regularización L2 (`C = 1`) ayuda a estabilizarlos.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.
